# Tutorial 2: Composition

This tutorial builds on [Tutorial 1 (counter)](counter.ipynb). We define an inverted pendulum environment and a controller neural network, then **compose** them into a closed-loop system using shared wires.

You'll learn:
1. How modules get their own fresh wires by default
2. How to create shared wire pairs and pass them to both modules
3. How `Module.parallel` merges modules that share wires

## The inverted pendulum environment

An inverted pendulum balanced upright. The full nonlinear dynamics:

$$\ddot\theta = \frac{g}{\ell}\,\sin\theta + \frac{\tau}{m\ell^2}$$

The positive gravitational term makes the upright equilibrium ($\theta = 0$) **unstable** — without active control, any perturbation grows. The controller must apply torque $\tau$ to keep it balanced.

Same `Env` contract as the counter: `__init__` sets spaces and state, `reset` initializes, `step` updates.

In [ ]:
import math
import torch
import torch.nn as nn
from gymnasium import spaces
from zrth import Env, NN, Module


class InvertedPendulumEnv(Env):
    """Inverted pendulum with full nonlinear dynamics."""

    def __init__(self, g=9.81, l=1.0, m=1.0, dt=0.05):
        super().__init__()

        self.action_space = spaces.Box(low=-2.0, high=2.0, shape=(1,))
        self.observation_space = spaces.Box(low=-10.0, high=10.0, shape=(1,))

        self.g = g
        self.l = l
        self.m = m
        self.dt = dt

        # Also set state variables here so the analyzer can infer their dtype
        self.theta = 0.1
        self.theta_dot = 0.0

    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        self.theta = 0.1
        self.theta_dot = 0.0

        observation = self.theta
        reward = 0.0
        terminated = False
        truncated = False
        return observation, reward, terminated, truncated

    def step(self, torque):
        accel = (self.g / self.l) * math.sin(self.theta) + torque / (self.m * self.l * self.l)
        self.theta_dot = self.theta_dot + accel * self.dt
        self.theta = self.theta + self.theta_dot * self.dt

        observation = self.theta
        reward = 0.0 - self.theta * self.theta - 0.1 * self.theta_dot * self.theta_dot
        terminated = self.theta > 3.14 or self.theta < -3.14
        truncated = False
        return observation, reward, terminated, truncated

Instantiate and inspect — same as in Tutorial 1.

In [ ]:
pendulum = InvertedPendulumEnv(g=9.81, l=1.0, m=1.0, dt=0.05)
print(pendulum)

## The controller

A small NN that observes $\theta$ and outputs a torque $\tau$. Architecture: $[1] \to 4 \to [1]$.

In [ ]:
class ControllerNN(NN):
    """NN controller: theta -> torque."""

    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(1, 4)
        self.fc2 = nn.Linear(4, 1)

    def forward(self, state):
        x = torch.relu(self.fc1(state))
        return self.fc2(x)

In [ ]:
controller = ControllerNN()
print(Module.__str__(controller))

## Composition with shared wires

By default, each `Env` and `NN` creates its own fresh wires. To connect them into a closed loop, we create wire pairs **up front** and pass them to both constructors:

- `Env` accepts `observation=` and `action=` kwargs
- `NN` accepts `extl=` and `intf=` kwargs

A wire pair is `[Wire(dtype), Wire(dtype)]` — one latched, one next. By passing the **same** pair as the pendulum's `observation` and the controller's `extl`, the env's observation output feeds the controller's input. Similarly, the controller's `intf` output feeds the env's `action` input.

`Module.parallel` then merges both modules. Because they share wires, the result is a single closed-loop system.

In [ ]:
from zrth import Wire, Float

# Shared wire pairs: pendulum observation = controller input
obs_wire = [Wire(Float(1)), Wire(Float(1))]
# Shared wire pairs: controller output = pendulum action
act_wire = [Wire(Float(1)), Wire(Float(1))]

pendulum = InvertedPendulumEnv(g=9.81, l=1.0, m=1.0, dt=0.05, observation=obs_wire, action=act_wire)
controller = ControllerNN(extl=obs_wire, intf=act_wire)

closed_loop = Module.parallel(pendulum, controller)
print(closed_loop)

## Training the controller

We train the controller using **REINFORCE** (policy gradient). The `InvertedPendulumEnv` and `ControllerNN` instances are fully functional — `pendulum.reset()` / `pendulum.step()` work as gym methods, and `controller(obs)` runs a PyTorch forward pass with autograd.

The controller outputs a mean torque $\mu$; we sample from $\mathcal{N}(\mu, \sigma^2)$ to explore. The loss is the standard REINFORCE estimator:

$$\nabla J = \mathbb{E}\left[\sum_t \nabla \log \pi(a_t | s_t) \cdot G_t\right]$$

where $G_t$ is the discounted return from step $t$.

In [ ]:
def compute_returns(rewards, gamma=0.99):
    """Compute discounted returns for each timestep."""
    returns = []
    G = 0.0
    for r in reversed(rewards):
        G = r + gamma * G
        returns.insert(0, G)
    return returns


pendulum = InvertedPendulumEnv(g=9.81, l=1.0, m=1.0, dt=0.05)
controller = ControllerNN()
optimizer = torch.optim.Adam(controller.parameters(), lr=0.001)

n_episodes = 300
max_steps = 200
sigma = 0.5  # exploration noise
gamma = 0.99

for episode in range(n_episodes):
    pendulum.reset()
    log_probs = []
    rewards = []

    for step in range(max_steps):
        state = torch.tensor([pendulum.theta], dtype=torch.float32)
        mu = controller(state.unsqueeze(0)).squeeze()
        dist = torch.distributions.Normal(mu, sigma)
        action = dist.sample().clamp(-2.0, 2.0)
        log_probs.append(dist.log_prob(action))

        _, reward, terminated, _ = pendulum.step(action.item())
        rewards.append(reward)
        if terminated:
            break

    # Policy gradient update
    returns = compute_returns(rewards, gamma)
    returns = torch.tensor(returns, dtype=torch.float32)
    returns = (returns - returns.mean()) / (returns.std() + 1e-8)  # baseline

    loss = torch.tensor(0.0)
    for lp, G in zip(log_probs, returns):
        loss = loss - lp * G

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    if episode % 50 == 0:
        total_reward = sum(rewards)
        print(f"episode {episode:3d}  steps = {len(rewards):3d}  reward = {total_reward:.2f}")

## Evaluating the trained controller

Run the trained controller deterministically (no exploration noise) and print $\theta$ at each step.

In [ ]:
pendulum.reset()

print(f"{'step':>4}  {'theta':>8}  {'theta_dot':>10}  {'torque':>8}  {'reward':>8}")
print("-" * 50)

with torch.no_grad():
    for step in range(20):
        state = torch.tensor([pendulum.theta], dtype=torch.float32)
        torque = controller(state.unsqueeze(0)).squeeze().item()
        torque = max(-2.0, min(2.0, torque))

        _, reward, terminated, _ = pendulum.step(torque)
        print(f"{step:4d}  {pendulum.theta:8.4f}  {pendulum.theta_dot:10.4f}  {torque:8.4f}  {reward:.4f}")
        if terminated:
            print("terminated!")
            break

## Running the composed module symbolically

The composed `closed_loop` module from earlier is a purely symbolic object — wires and terms, no Python code. To actually *run* it, we wrap it back as a gym environment using `Env(module)`. This evaluates the symbolic terms numerically via the built-in interpreter.

This completes the round-trip: **define** → **compose** → **run symbolically**.

In [ ]:
import numpy as np

# closed_loop was composed earlier via Module.parallel(pendulum, controller)
# Wrap it as a runnable gym.Env, this uses the symbolic term interpreter
runnable = Env(closed_loop)

obs, info = runnable.reset()
print(f"{'step':>4}  {'obs':>10}")
print("-" * 18)

for step in range(20):
    # The closed-loop system has no external inputs, action is internal
    obs, reward, terminated, truncated, info = runnable.step(np.zeros(1))
    print(f"{step:4d}  {float(obs[0]):10.4f}")
    if terminated:
        print("terminated!")
        break